# Etapa de GPU — embeddings, toxicidade e sentimento (9 países)

Tudo o que precisa de GPU num notebook só. Saídas (só números, sem letras):

| Arquivo | Conteúdo |
|---|---|
| `embeddings/BAAI_bge_m3__original.npz`, `__traducao_en.npz` | BGE-M3 (principal), letra original e traduzida |
| `embeddings/sentence_transformers_LaBSE__*.npz` | LaBSE (robustez; reaproveita o cache antigo) |
| `embeddings/temas__*.npz` | vetores das descrições dos 10 temas (classificação por similaridade) |
| `toxicidade.csv` | Detoxify: `multilingual` na original (en, es, pt, fr) e `unbiased` na tradução (todas) |
| `sentimento.csv` | XLM-T (XLM-R treinado para sentimento), na original e na tradução |

**Como rodar no Kaggle (T4, grátis):**
1. *Datasets → New dataset*: suba `comum.py`, `finalData.csv` e a pasta `dados/` (os caches antigos de
   embeddings e toxicidade são reaproveitados).
2. *New notebook → File → Import notebook* com este arquivo; em *Settings*: **Accelerator = GPU T4** e **Internet = On**
   (precisa de telefone verificado); *Add input* = o dataset do passo 1.
3. *Run all* (~1 h). No fim, baixe `/kaggle/working/saida.zip` e descompacte dentro de `dados/` do repositório.

Fora do Kaggle, o notebook usa a pasta do repositório e grava direto em `dados/`.
Teste rápido em CPU: `SAIDA=/tmp/teste GPU_LIMITE=2 EMB_PRINCIPAL=sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 SEM_LABSE=1`.

In [1]:
import os, sys, re, glob, time, hashlib, shutil, subprocess
from pathlib import Path
import numpy as np
import pandas as pd

NO_KAGGLE = Path("/kaggle/working").exists()
if NO_KAGGLE:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "detoxify", "sentencepiece", "protobuf"], check=False)

# ---------- onde está o repositório (comum.py + finalData.csv) e onde gravar
def acha_repo():
    if os.environ.get("REPO"):
        return Path(os.environ["REPO"])
    if Path("comum.py").exists():
        return Path(".").resolve()
    achados = glob.glob("/kaggle/input/**/comum.py", recursive=True)
    if not achados:
        raise FileNotFoundError("comum.py não encontrado: adicione o dataset com o repositório como input")
    return Path(achados[0]).parent

REPO = acha_repo()
OUT = Path(os.environ["SAIDA"]) if os.environ.get("SAIDA") else (Path("/kaggle/working/saida") if NO_KAGGLE else REPO / "dados")
PASTA_EMB, PASTA_TMP = OUT / "embeddings", OUT / "cache"
PASTA_EMB.mkdir(parents=True, exist_ok=True); PASTA_TMP.mkdir(parents=True, exist_ok=True)
if OUT != REPO / "dados":                     # Kaggle: copia os caches antigos para a pasta gravável
    for a in glob.glob(str(REPO / "dados/embeddings/*.npz")):
        shutil.copy(a, PASTA_EMB)
    for a in glob.glob(str(REPO / "dados/cache/tox_*.csv")):
        shutil.copy(a, PASTA_TMP)
sys.path.insert(0, str(REPO))
from comum import carrega_top

import torch
DEV = "cuda" if torch.cuda.is_available() else "cpu"
print("repositório:", REPO, "| saída:", OUT, "| dispositivo:", DEV,
      torch.cuda.get_device_name(0) if DEV == "cuda" else "")

# ---------- parâmetros
EMB_PRINCIPAL = os.environ.get("EMB_PRINCIPAL", "BAAI/bge-m3")
EMB_ROBUSTEZ  = "sentence-transformers/LaBSE"
SEM_LABSE     = os.environ.get("SEM_LABSE") == "1"
SENT_MODELO   = os.environ.get("SENT_MODELO", "cardiffnlp/twitter-xlm-roberta-base-sentiment")
LIMITE        = int(os.environ.get("GPU_LIMITE", 0))        # > 0: só N faixas de cada idioma (teste)
BATCH         = 32 if DEV == "cuda" else 8
MAX_TOK_EMB   = 1024                                         # BGE-M3 aceita 8192; letras maiores são divididas
MAX_TOK_CLS   = 400                                          # Detoxify e XLM-T: limite de 512 tokens
LINGUAS_MULTI = {"en", "es", "pt", "fr"}                     # línguas do Detoxify multilingual presentes nos dados

repositório: /home/besome/Documentos/Coding/socialComp/SocialComp | saída: /home/besome/Documentos/Coding/socialComp/SocialComp/dados | dispositivo: cuda NVIDIA GeForce RTX 4050 Laptop GPU


In [2]:
# todas as faixas do finalData.csv (top 120 de cada célula), para não precisar voltar à GPU se o recorte mudar
d = carrega_top(n=10**6, derivadas=False)
faixas = d.drop_duplicates("track_id").set_index("track_id")[["idioma", "letra_limpa", "letra_en_limpa"]]
faixas["letra_en_limpa"] = faixas["letra_en_limpa"].fillna(faixas["letra_limpa"])
if LIMITE:
    faixas = faixas.groupby("idioma", group_keys=False).head(LIMITE)
    print(f"MODO TESTE: {LIMITE} faixas por idioma")
print(f"{len(faixas)} faixas | idiomas: {faixas['idioma'].value_counts().to_dict()}")

4727 faixas | idiomas: {'en': 1743, 'pt': 852, 'es': 734, 'fr': 557, 'ja': 448, 'de': 341, 'outro': 52}


## 1. Embeddings

Letras maiores que o limite do modelo são divididas em trechos (respeitando as linhas); os vetores dos trechos são normalizados, promediados e renormalizados. O cache é por `track_id` + hash do texto, o mesmo esquema do `pipelineAnalise`, então os embeddings LaBSE já calculados são reaproveitados.

In [3]:
def divide_trechos(texto, tok, max_tok):
    linhas = [l for l in str(texto).split("\n") if l.strip()]
    trechos, atual, n = [], [], 0
    for l in linhas:
        k = len(tok.tokenize(l)) + 1
        if atual and n + k > max_tok:
            trechos.append("\n".join(atual)); atual, n = [], 0
        atual.append(l); n += k
    if atual:
        trechos.append("\n".join(atual))
    return trechos or [str(texto)]

slug = lambda s: re.sub(r"[^\w]+", "_", s).strip("_")
chave = lambda tid, t: f"{tid}:{hashlib.md5(t.encode('utf-8')).hexdigest()[:10]}"

def carrega_st(nome):
    from sentence_transformers import SentenceTransformer
    m = SentenceTransformer(nome, device=DEV)
    m.max_seq_length = min(m.max_seq_length, MAX_TOK_EMB)
    if DEV == "cuda":
        m.half()
    return m

def gera_embeddings(m, nome_modelo, textos, nome):
    arq = PASTA_EMB / f"{slug(nome_modelo)}__{nome}.npz"
    chaves = [chave(t, x) for t, x in textos.items()]
    cache = {}
    if arq.exists():
        z = np.load(arq)
        cache = dict(zip(z["chaves"].tolist(), z["emb"]))
    falta = [(c, x) for c, x in zip(chaves, textos.values) if c not in cache]
    print(f"[{slug(nome_modelo)}/{nome}] {len(chaves) - len(falta)} no cache, {len(falta)} a calcular")
    if falta:
        max_tok = m.max_seq_length - 8
        pedacos, dono = [], []
        for c, x in falta:
            for b in divide_trechos(x, m.tokenizer, max_tok):
                pedacos.append(b); dono.append(c)
        t0 = time.time()
        V = m.encode(pedacos, batch_size=BATCH, normalize_embeddings=True, show_progress_bar=True, convert_to_numpy=True)
        V = V.astype(np.float32); dono = np.array(dono)
        for c, _ in falta:
            v = V[dono == c].mean(axis=0)
            cache[c] = v / np.linalg.norm(v)
        ks = list(cache)
        np.savez_compressed(arq, chaves=np.array(ks), emb=np.stack([cache[k] for k in ks]).astype(np.float32))
        print(f"   {len(pedacos)} trechos em {time.time() - t0:.0f}s -> {arq.name}")

# 10 temas da taxonomia fixa, descritos em inglês (são comparados com a letra traduzida)
TEMAS = {
    "amor": "a love song about romance, being in love, devotion and desire for a partner",
    "termino": "heartbreak, a breakup, missing an ex, betrayal and lost love",
    "festa": "party, dancing, the club, drinking and having fun tonight",
    "dinheiro_status": "money, wealth, luxury, fame, success and showing off",
    "sexo": "explicit sexual desire, sex and seduction",
    "violencia_rua": "street life, crime, guns, drugs, gangs and violence",
    "superacao": "self-empowerment, overcoming struggles, confidence and ambition",
    "nostalgia": "nostalgia, memories, longing for the past, sadness and loneliness",
    "espiritualidade": "faith, God, religion, prayer and spirituality",
    "critica_social": "social criticism, politics, injustice, society and protest",
}

def embeddings_de(nome_modelo):
    m = carrega_st(nome_modelo)
    print(f"{nome_modelo} em {DEV} | max_seq_length = {m.max_seq_length}")
    gera_embeddings(m, nome_modelo, faixas["letra_limpa"], "original")
    gera_embeddings(m, nome_modelo, faixas["letra_en_limpa"], "traducao_en")
    T = m.encode(list(TEMAS.values()), normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
    np.savez_compressed(PASTA_EMB / f"temas__{slug(nome_modelo)}.npz", temas=np.array(list(TEMAS)), emb=T)
    del m
    if DEV == "cuda":
        torch.cuda.empty_cache()

embeddings_de(EMB_PRINCIPAL)
if not SEM_LABSE:
    embeddings_de(EMB_ROBUSTEZ)

/home/besome/Documentos/Coding/socialComp/SocialComp/.venv/lib64/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|                                                                                                                                                 | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 391/391 [00:00<00:00, 45941.48it/s]

BAAI/bge-m3 em cuda | max_seq_length = 1024
[BAAI_bge_m3/original] 0 no cache, 4727 a calcular


Batches:   0%|                                                                                                                                                         | 0/163 [00:00<?, ?it/s]

Batches:   1%|▉                                                                                                                                                | 1/163 [00:01<04:37,  1.71s/it]

... (643 outputs left out)

Batches:  97%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████▊   | 327/336 [00:26<00:00, 35.35it/s]

Batches: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 336/336 [00:26<00:00, 49.65it/s]

Batches: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 336/336 [00:26<00:00, 12.55it/s]

   10752 trechos em 27s -> sentence_transformers_LaBSE__traducao_en.npz


## 2. Toxicidade (Detoxify)

O `multilingual` (XLM-R) roda na letra original quando a língua está no treino dele (en, es, pt, fr); alemão e japonês ficam com o escore do `unbiased` (RoBERTa) na tradução. Por música: média dos trechos ponderada pelo tamanho e máximo (a música tem pelo menos um trecho pesado?).

In [4]:
_tox = {}

def carrega_detoxify(tipo):
    if tipo not in _tox:
        from detoxify import Detoxify
        _tox[tipo] = Detoxify(tipo, device=DEV)
    return _tox[tipo]

def pontua(textos, tipo, nome, bloco=200):
    arq = PASTA_TMP / f"tox_{nome}_{tipo}.csv"
    textos = textos.dropna()
    chaves = pd.Series([chave(t, x) for t, x in textos.items()], index=textos.index)
    cache = pd.read_csv(arq).set_index("chave") if arq.exists() else pd.DataFrame()
    falta = [t for t in textos.index if chaves[t] not in cache.index]
    print(f"[tox {nome}/{tipo}] {len(textos) - len(falta)} no cache, {len(falta)} a calcular")
    if falta:
        m = carrega_detoxify(tipo); t0 = time.time()
        for ini in range(0, len(falta), bloco):
            lote = falta[ini:ini + bloco]
            trechos, dono, ntok = [], [], []
            for t in lote:
                for b in divide_trechos(textos[t], m.tokenizer, MAX_TOK_CLS):
                    trechos.append(b); dono.append(t); ntok.append(len(m.tokenizer.tokenize(b)))
            ordem = np.argsort(ntok)
            esc = {c: np.empty(len(trechos)) for c in m.class_names}
            for i in range(0, len(ordem), BATCH):
                ix = ordem[i:i + BATCH]
                out = m.predict([trechos[j] for j in ix])
                for c in m.class_names:
                    esc[c][ix] = np.asarray(out[c], dtype=float)
            df = pd.DataFrame({"track_id": dono, "ntok": ntok, **esc})
            linhas = []
            for t, g in df.groupby("track_id", sort=False):
                r = {"chave": chaves[t], "track_id": t, "trechos": len(g)}
                for c in m.class_names:
                    r[f"{c}_media"] = np.average(g[c], weights=g["ntok"]); r[f"{c}_max"] = g[c].max()
                linhas.append(r)
            cache = pd.concat([cache, pd.DataFrame(linhas).set_index("chave")])
            cache.to_csv(arq)
            print(f"   {min(ini + bloco, len(falta))}/{len(falta)} | {time.time() - t0:.0f}s")
    return cache.loc[chaves.values].set_index("track_id").drop(columns=["trechos"])

multi = faixas["idioma"].isin(LINGUAS_MULTI)
tox_o = pontua(faixas.loc[multi, "letra_limpa"], "multilingual", "original").add_prefix("orig_")
tox_e = pontua(faixas["letra_en_limpa"], "unbiased", "traduzida").add_prefix("en_")
tox = faixas[["idioma"]].join(tox_o).join(tox_e)
tox["tox"] = np.where(multi, tox["orig_toxicity_media"], tox["en_toxicity_media"])
tox["tox_max"] = np.where(multi, tox["orig_toxicity_max"], tox["en_toxicity_max"])
tox["toxica"] = tox["tox_max"] >= 0.5
tox["fonte_tox"] = np.where(multi, "original/multilingual", "traduzida/unbiased")
tox.to_csv(OUT / "toxicidade.csv")
_tox.clear()
if DEV == "cuda":
    torch.cuda.empty_cache()
print(tox.groupby("idioma")[["tox", "tox_max", "toxica"]].mean().round(3))

[tox original/multilingual] 0 no cache, 3886 a calcular
Downloading: "https://github.com/unitaryai/detoxify/releases/download/v0.4-alpha/multilingual_debiased-0b549669.ckpt" to /home/besome/.cache/torch/hub/checkpoints/multilingual_debiased-0b549669.ckpt


  0%|                                                                                                                                    | 0.00/1.04G [00:00<?, ?B/s]

  0%|▏                                                                                                                          | 1.88M/1.04G [00:00<00:59, 18.8MB/s]

  1%|▊                                                                                                                          | 6.62M/1.04G [00:00<00:30, 36.7MB/s]

  1%|█▏                                                                                                                         | 10.2M/1.04G [00:00<00:35, 30.9MB/s]

  1%|█▌                                                                                                                         | 13.4M/1.04G [00:00<00:36, 30.0MB/s]

... (627 outputs left out)

   4400/4727 | 128s


   4600/4727 | 133s


   4727/4727 | 137s


          tox  tox_max  toxica
idioma                        
de      0.327    0.439   0.457
en      0.320    0.395   0.390
es      0.397    0.470   0.469
fr      0.474    0.623   0.645
ja      0.150    0.207   0.190
outro   0.149    0.236   0.192
pt      0.482    0.544   0.558


## 3. Sentimento (XLM-T)

XLM-R ajustado para sentimento em 8 línguas (Barbieri et al., 2022). Roda na letra original e na tradução, para ver quanto a tradução muda o tom. Por música: probabilidades médias dos trechos, ponderadas pelo tamanho; `sent` = P(positivo) − P(negativo).

In [5]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

tok = AutoTokenizer.from_pretrained(SENT_MODELO)
mod = AutoModelForSequenceClassification.from_pretrained(SENT_MODELO).to(DEV).eval()
if DEV == "cuda":
    mod.half()
rotulos = [mod.config.id2label[i].lower() for i in range(mod.config.num_labels)]
print("rótulos:", rotulos)

def sentimento(textos, nome):
    arq = PASTA_TMP / f"sent_{nome}.csv"
    chaves = pd.Series([chave(t, x) for t, x in textos.items()], index=textos.index)
    cache = pd.read_csv(arq).set_index("chave") if arq.exists() else pd.DataFrame()
    falta = [t for t in textos.index if chaves[t] not in cache.index]
    print(f"[sent {nome}] {len(textos) - len(falta)} no cache, {len(falta)} a calcular")
    t0 = time.time()
    for ini in range(0, len(falta), 200):
        lote = falta[ini:ini + 200]
        trechos, dono, ntok = [], [], []
        for t in lote:
            for b in divide_trechos(textos[t], tok, MAX_TOK_CLS):
                trechos.append(b); dono.append(t); ntok.append(len(tok.tokenize(b)))
        ordem = np.argsort(ntok); P = np.empty((len(trechos), len(rotulos)))
        for i in range(0, len(ordem), BATCH):
            ix = ordem[i:i + BATCH]
            enc = tok([trechos[j] for j in ix], padding=True, truncation=True, max_length=512, return_tensors="pt").to(DEV)
            with torch.no_grad():
                P[ix] = torch.softmax(mod(**enc).logits.float(), dim=-1).cpu().numpy()
        df = pd.DataFrame(P, columns=rotulos).assign(track_id=dono, ntok=ntok)
        linhas = []
        for t, g in df.groupby("track_id", sort=False):
            r = {"chave": chaves[t], "track_id": t}
            r.update({f"p_{c}": np.average(g[c], weights=g["ntok"]) for c in rotulos})
            linhas.append(r)
        cache = pd.concat([cache, pd.DataFrame(linhas).set_index("chave")])
        cache.to_csv(arq)
        print(f"   {min(ini + 200, len(falta))}/{len(falta)} | {time.time() - t0:.0f}s")
    r = cache.loc[chaves.values].set_index("track_id")
    r["sent"] = r["p_positive"] - r["p_negative"]
    return r

sent = (faixas[["idioma"]]
        .join(sentimento(faixas["letra_limpa"], "original").add_prefix("orig_"))
        .join(sentimento(faixas["letra_en_limpa"], "traduzida").add_prefix("en_")))
sent.to_csv(OUT / "sentimento.csv")
print(sent.groupby("idioma")[["orig_sent", "en_sent"]].mean().round(3))

Loading weights:   0%|                                                                                                                       | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████| 201/201 [00:00<00:00, 43564.24it/s]

rótulos: ['negative', 'neutral', 'positive']
[sent original] 0 no cache, 4727 a calcular


   200/4727 | 2s


   400/4727 | 4s


   600/4727 | 6s


... (41 outputs left out)

   4200/4727 | 44s


   4400/4727 | 46s


   4600/4727 | 47s


   4727/4727 | 49s
        orig_sent  en_sent
idioma                    
de         -0.063   -0.058
en          0.010    0.010
es          0.018   -0.086
fr         -0.003   -0.129
ja          0.027   -0.125
outro       0.036    0.012
pt         -0.052   -0.073


In [6]:
# resumo e pacote para download
for a in sorted(glob.glob(str(OUT / "**/*"), recursive=True)):
    if Path(a).is_file():
        print(f"{Path(a).stat().st_size / 1e6:8.1f} MB  {Path(a).relative_to(OUT)}")
if NO_KAGGLE:
    shutil.make_archive("/kaggle/working/saida", "zip", OUT)
    print("baixe /kaggle/working/saida.zip e descompacte dentro de dados/ do repositório")

    14.3 MB  cache/mgd_entradas_diarias.csv.gz
     0.8 MB  cache/origem_artistas.csv
     4.9 MB  cache/primeira_aparicao.csv
     0.5 MB  cache/sent_original.csv
     0.5 MB  cache/sent_traduzida.csv
     1.4 MB  cache/tox_original_multilingual.csv
     1.7 MB  cache/tox_traduzida_unbiased.csv
    17.3 MB  embeddings/BAAI_bge_m3__original.npz
    17.4 MB  embeddings/BAAI_bge_m3__traducao_en.npz
    14.0 MB  embeddings/sentence_transformers_LaBSE__original.npz
    14.0 MB  embeddings/sentence_transformers_LaBSE__traducao_en.npz
     0.0 MB  embeddings/temas__BAAI_bge_m3.npz
     0.0 MB  embeddings/temas__sentence_transformers_LaBSE.npz
    17.3 MB  letras_cache.jsonl
    17.3 MB  letras_cache.jsonl.bak
     0.9 MB  sentimento.csv
     3.0 MB  toxicidade.csv
